# 03 – Geology and surface analysis of Heard Island

Prepares the input layers for the geology analysis:

1. **Study area** – read the bounding box exported from QGIS and build a common 25 m grid
2. **Input data** – geology map and 10 m DEM from the Australian Antarctic Data Centre
3. **Terrain derivatives** – slope, aspect and hillshade from the DEM
4. **Helpers** – raster reclassification and reproject/clip tools
5. **Apply** – clip and reproject layers onto the project grid (following on from earlier steps with EPSG:32743)

Setup

In [1]:
import os
import uuid
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import shapely
from shapely.geometry import box
from osgeo import gdal, ogr

# PyQGIS (requires the QGIS Python environment to be available/initialised)
import processing
from qgis.core import QgsCoordinateReferenceSystem, QgsVectorLayer

gdal.UseExceptions()

# ---- Project configuration: edit these, not the code below ----
def find_repo_root(start=None):
    """Walk up from the notebook's folder until a .git folder is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in (start, *start.parents):
        if (folder / ".git").exists():
            return folder
    raise FileNotFoundError("No .git folder found above " + str(start))

REPO_ROOT  = find_repo_root()
inpath     = REPO_ROOT / "Input_Data"
outpath    = REPO_ROOT / "Output_Data"
TEMP_DIR   = outpath / "temp"            # scratch space for intermediate vectors
TARGET_CRS = "EPSG:7855"                 # GDA2020 / MGA zone 55
CELL_SIZE  = 25                          # metres; every raster shares this grid

# Study area (area of interest) in WGS84 lon/lat: [xmin, ymin, xmax, ymax]
aoi_wgs84 = [73.45, -53.25, 73.85, -52.95]

GEOLOGY_ZIP = inpath / "Geology" / "Heard_Island_Geology_Map.zip"
DEM_ZIP     = inpath / "Rectified_RADARSAT_image_of_Heard_Island_acquired_January_2002.zip"

for d in (outpath, TEMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

Application path not initialized


Unzipper for downloaded zipped datasets and deal with shapefiles with unclosed polygons.

In [2]:
def extract_if_zip(local_file, output_dir):
    """Unzip `local_file` into <output_dir>/extracted and return that folder.
    If it isn't a zip, return the folder it already lives in."""
    os.makedirs(output_dir, exist_ok=True)
    if zipfile.is_zipfile(local_file):
        extract_to = os.path.join(output_dir, "extracted")
        with zipfile.ZipFile(local_file) as z:
            z.extractall(extract_to)
        return extract_to
    return os.path.dirname(local_file)

def read_vector_closing_rings(path):
    """Read a vector file with OGR and close any unclosed polygon rings.
    geopandas/shapely refuse to load rings whose first and last points differ
    (GEOSException: 'Points of LinearRing do not form a closed linestring')."""
    ds = ogr.Open(str(path))
    layer = ds.GetLayer(0)
    defn = layer.GetLayerDefn()
    fields = [defn.GetFieldDefn(i).GetName() for i in range(defn.GetFieldCount())]
    srs = layer.GetSpatialRef()

    records, geoms = [], []
    for feat in layer:
        geom = feat.GetGeometryRef()
        if geom is not None:
            geom = geom.Clone()
            geom.CloseRings()            # no-op for lines/points
            geoms.append(shapely.from_wkb(bytes(geom.ExportToWkb())))
        else:
            geoms.append(None)
        records.append({f: feat.GetField(f) for f in fields})

    return gpd.GeoDataFrame(records, geometry=geoms, crs=srs.ExportToWkt() if srs else None)


def read_vector_safe(path):
    """geopandas read, falling back to ring-closing + validity repair for broken shapefiles."""
    try:
        return gpd.read_file(path)
    except shapely.errors.GEOSException as e:
        print(f"geopandas could not read {os.path.basename(path)} ({e}); closing rings with OGR")
        gdf = read_vector_closing_rings(path)
        n_bad = (~gdf.is_valid).sum()
        if n_bad:
            print(f"Repairing {n_bad} invalid geometries with make_valid()")
            gdf["geometry"] = gdf.make_valid()
        return gdf

Study area and project grid
The AOI is defined by `aoi_wgs84` (lon/lat) in the setup cell. It is reprojected to the project CRS and
snapped to the cell size so every raster ends up with identical row/column counts.

In [3]:
# Build the study area polygon from the AOI coordinates (WGS84 lon/lat)
study_area = gpd.GeoDataFrame(geometry=[box(*aoi_wgs84)], crs="EPSG:4326")

# STAC bounding boxes need to be WGS84 lon/lat: [xmin, ymin, xmax, ymax]
bbox = list(aoi_wgs84)

# Project CRS version of the study area, saved so PyQGIS can use it as a clip layer
study_area_proj = study_area.to_crs(TARGET_CRS)
aoi_path = outpath / "aoi_projected.gpkg"
study_area_proj.to_file(aoi_path, driver="GPKG")
bounding_vector = QgsVectorLayer(str(aoi_path), "AOI")   # for PyQGIS clip tools

xmin, ymin, xmax, ymax = study_area_proj.total_bounds

# Snap the extent outward to the cell size
xmin = xmin - (xmin % CELL_SIZE)
ymin = ymin - (ymin % CELL_SIZE)
xmax = xmin + CELL_SIZE * (((xmax - xmin) // CELL_SIZE) + 1)
ymax = ymin + CELL_SIZE * (((ymax - ymin) // CELL_SIZE) + 1)

cols = int(round((xmax - xmin) / CELL_SIZE))
rows = int(round((ymax - ymin) / CELL_SIZE))

# String format QGIS Processing expects for EXTENT parameters: "xmin,xmax,ymin,ymax [EPSG:code]"
global_extent_str = f"{xmin},{xmax},{ymin},{ymax} [{TARGET_CRS}]"

print(global_extent_str, cols, rows)

-3711100.0,-3661425.0,1332775.0,1377725.0 [EPSG:7855] 1987 1798


Application path not initialized
Application path not initialized
Application path not initialized
Application path not initialized
Application path not initialized
Application path not initialized


Input data.
First geology map from the Australian Antarctic Data Cetre

In [4]:
GEOLOGY_DIR = "heard_island_geology"
search_dir = extract_if_zip(GEOLOGY_ZIP, GEOLOGY_DIR)

# Find whatever spatial files are in the archive
shapefiles, gdbs = [], []
for root, dirs, files in os.walk(search_dir):
    gdbs += [os.path.join(root, d) for d in dirs if d.lower().endswith(".gdb")]
    shapefiles += [os.path.join(root, f) for f in files if f.lower().endswith(".shp")]

print("Shapefiles found:", shapefiles)
print("Geodatabases found:", gdbs)

# The archive holds several shapefiles (contours, outline, dates...), so pick the geology polygons by name
geology_shp = next((s for s in shapefiles if "geology" in os.path.basename(s).lower()), None)

if geology_shp:
    gdf = read_vector_safe(geology_shp)
    print("Using:", geology_shp)
    print(gdf.head())
    print(f"CRS: {gdf.crs}")
elif gdbs:
    import fiona
    layers = fiona.listlayers(gdbs[0])
    print("Layers in geodatabase:", layers)
    gdf = gpd.read_file(gdbs[0], layer=layers[0])
    print(gdf.head())

Shapefiles found: ['heard_island_geology/extracted/Big Ben Lines.shp', 'heard_island_geology/extracted/Ar_dates.shp', 'heard_island_geology/extracted/HI_Geology_poly.shp', 'heard_island_geology/extracted/hi_contours.shp', 'heard_island_geology/extracted/HI_Outline.shp', 'heard_island_geology/extracted/contours_20m.shp', 'heard_island_geology/extracted/contours_20m_2.shp']
Geodatabases found: ['heard_island_geology/extracted/AADC_Fox_Heard.gdb']
geopandas could not read HI_Geology_poly.shp (IllegalArgumentException: Points of LinearRing do not form a closed linestring); closing rings with OGR


/Users/jamessampson/miniconda3/envs/KGG375_AT2/lib/python3.12/site-packages/pyogrio/raw.py:198: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return ogr_read(
/Users/jamessampson/miniconda3/envs/KGG375_AT2/lib/python3.12/site-packages/osgeo/ogr.py:1036: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return _ogr.Layer_GetNextFeature(self, *args)


GEOSException: IllegalArgumentException: Points of LinearRing do not form a closed linestring

DEM. 10m from the Australian Antarctic Data Centre

In [ ]:
DEM_DIR = "heard_island_dem"
search_dir = extract_if_zip(DEM_ZIP, DEM_DIR)

# Find raster files (case-insensitive extension match)
rasters = [os.path.join(root, f)
           for root, _, files in os.walk(search_dir)
           for f in files if f.lower().endswith((".tif", ".tiff"))]
print("Raster files found:", rasters)

# Prefer the 16-bit version (real elevation values) over any 8-bit display version
dem_path = next((r for r in rasters if "16bit" in r.lower()), rasters[0] if rasters else None)

if dem_path:
    print("Using:", dem_path)
    with rasterio.open(dem_path) as dem:
        elevation = dem.read(1)
        print(f"CRS: {dem.crs}")
        print(f"Shape: {elevation.shape}")
        print(f"Min/Max elevation: {elevation.min()} / {elevation.max()}")
else:
    print("No raster found.")

Compute terrain derivatives from the DEM
All using GDAL DEM processing

In [ ]:
def slope(dem_path, out_path, units="degree", scale=None, compute_edges=True):
    """Slope raster. units: 'degree' or 'percent'."""
    opts = gdal.DEMProcessingOptions(
        slopeFormat=units,
        scale=scale,
        computeEdges=compute_edges,
        creationOptions=["COMPRESS=LZW"],
    )
    gdal.DEMProcessing(out_path, dem_path, "slope", options=opts)
    return out_path


def aspect(dem_path, out_path, zero_for_flat=False, compute_edges=True):
    """Aspect in degrees clockwise from north. Flat cells are nodata (-9999)
    unless zero_for_flat=True."""
    opts = gdal.DEMProcessingOptions(
        zeroForFlat=zero_for_flat,
        computeEdges=compute_edges,
        creationOptions=["COMPRESS=LZW"],
    )
    gdal.DEMProcessing(out_path, dem_path, "aspect", options=opts)
    return out_path


def hillshade(dem_path, out_path, azimuth=315, altitude=45, z_factor=1.0,
              scale=None, multi_directional=False, combined=False,
              compute_edges=True):
    """Hillshade (0-255). Set multi_directional=True for a softer,
    multi-light-source look that works well in rugged terrain."""
    opts = gdal.DEMProcessingOptions(
        azimuth=azimuth,
        altitude=altitude,
        zFactor=z_factor,
        scale=scale,
        multiDirectional=multi_directional,
        combined=combined,
        computeEdges=compute_edges,
        creationOptions=["COMPRESS=LZW"],
    )
    gdal.DEMProcessing(out_path, dem_path, "hillshade", options=opts)
    return out_path

Plot all of these derivatives

In [ ]:
slope_path     = slope(dem_path, os.path.join(DEM_DIR, "slope.tif"))
aspect_path    = aspect(dem_path, os.path.join(DEM_DIR, "aspect.tif"))
hillshade_path = hillshade(dem_path, os.path.join(DEM_DIR, "hillshade.tif"))


def read_masked(path):
    with rasterio.open(path) as src:
        return src.read(1, masked=True)   # nodata automatically masked


fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(read_masked(hillshade_path), cmap="gray");  ax[0].set_title("Hillshade")
ax[1].imshow(read_masked(slope_path), cmap="magma");     ax[1].set_title("Slope (°)")
ax[2].imshow(read_masked(aspect_path), cmap="twilight"); ax[2].set_title("Aspect (°)")
plt.show()

Reclassify rasters

In [ ]:
def reclassify_raster(input_raster, output_raster, classification_table, range_boundaries):
    """
    Reclassify a raster using a classification table (native:reclassifybytable).

    Example classification table (min, max, new value):
        table = [
            0.0, 0.2, 1,        # Very low
            0.2, 0.4, 2,        # Low
            0.4, 0.6, 3,        # Moderate
            0.6, 0.8, 4,        # High
            0.8, 1.000001, 5,   # Very high
        ]
    """
    result = processing.run("native:reclassifybytable", {
        'INPUT_RASTER': str(input_raster),
        'RASTER_BAND': 1,            # keep this constant
        'TABLE': classification_table,
        'NO_DATA': -9999,
        'RANGE_BOUNDARIES': range_boundaries,
        'NODATA_FOR_MISSING': False,
        'OUTPUT': str(output_raster),
    })
    return result["OUTPUT"]

Project and clip layers to study area

In [ ]:
GRID = {
    "extent_str": global_extent_str,
    "cols": cols,
    "rows": rows,
    "crs": QgsCoordinateReferenceSystem(TARGET_CRS),
}
print(GRID)


def snap_to_grid(input_path, output_path, grid, resampling_method=0):
    """Force any raster onto the project's pixel grid, regardless of how it was created."""
    result = processing.run("gdal:warpreproject", {
        'INPUT': str(input_path),
        'TARGET_CRS': grid["crs"],
        'RESAMPLING': resampling_method,
        'TARGET_EXTENT': grid["extent_str"],
        'TARGET_EXTENT_CRS': grid["crs"],
        'EXTRA': f'-ts {grid["cols"]} {grid["rows"]}',
        'OUTPUT': str(output_path),
    })
    return result['OUTPUT']


def project_and_clip(input_layer, output_path, bounding_box, layer_type="raster",
                     resampling_method=None, cell_size=CELL_SIZE, target_crs=TARGET_CRS, grid=None):
    """
    Reproject and clip a layer to the project's consistent geometry and projection.

    raster: reproject to `target_crs` at `cell_size`, then snap onto `grid`.
    vector: clip to `bounding_box`, then reproject to `target_crs`.
    """
    target_crs = QgsCoordinateReferenceSystem(target_crs)
    output_path = str(output_path)

    if layer_type == "raster":
        reproject_result = processing.run("gdal:warpreproject", {
            'INPUT': input_layer,
            'TARGET_CRS': target_crs,
            'RESAMPLING': resampling_method,
            'TARGET_RESOLUTION': cell_size,
            'OUTPUT': 'TEMPORARY_OUTPUT',
        })
        return snap_to_grid(reproject_result['OUTPUT'], output_path, grid, resampling_method)

    elif layer_type == "vector":
        temp_clip_path = str(TEMP_DIR / f"clip_{uuid.uuid4().hex}.gpkg")

        try:
            clip_result = processing.run("native:clip", {
                'INPUT': input_layer,
                'OVERLAY': bounding_box,
                'OUTPUT': temp_clip_path,
            })
        except Exception as e:
            print(f"CLIP FAILED for {output_path}: {e}")
            raise

        try:
            reproject_result = processing.run("native:reprojectlayer", {
                'INPUT': clip_result['OUTPUT'],
                'TARGET_CRS': target_crs,
                'OUTPUT': output_path,
            })
        except Exception as e:
            print(f"REPROJECT FAILED for {output_path}: {e}")
            raise

        return reproject_result['OUTPUT']

    else:
        raise ValueError("layer_type must be 'raster' or 'vector'")

Clip and reproject each of these datasets into the project grid

In [ ]:
# Geology units (250k) are shapefiles, so load them into QGIS with QgsVectorLayer
input_geology_file = inpath / "geology250k" / "geology_units250k.shp"
geology_units250_qgis = QgsVectorLayer(str(input_geology_file), "MRT Geology Layer")

# DEM: bilinear resampling (= 1 in GDAL)
dem_raster = dem_path
dem_clipped = project_and_clip(
    dem_raster,
    outpath / "DEM_25m_7855_clipped.tif",
    bounding_vector,
    layer_type="raster",
    resampling_method=1,
    grid=GRID,
)

Clip slope raster

In [ ]:
slope_result = processing.run("gdal:slope", {
    'INPUT': str(dem_clipped),
    'BAND': 1,
    'SCALE': 1,
    'AS_PERCENT': False,
    'COMPUTE_EDGES': False,
    'ZEVENBERGEN': False,
    'OPTIONS': '',
    'EXTRA': '',
    'OUTPUT': 'TEMPORARY_OUTPUT',
})
slope_clipped = slope_result["OUTPUT"]